In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

warnings.filterwarnings("ignore")

# Ensure output directories exist
os.makedirs("model", exist_ok=True)

# ==============================================================================
# 1. LOAD DATASET & INITIAL INSPECTION
# ==============================================================================
print("=" * 70)
print("1. DATASET OVERVIEW")
print("=" * 70)

df = pd.read_csv("adult.csv")
print(f"Initial Shape: {df.shape[0]} rows, {df.shape[1]} columns")

# Strip whitespace from string columns and convert '?' to NaN
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()
df.replace("?", np.nan, inplace=True)

print("\nMissing values per column (after converting '?' to NaN):")
print(df.isna().sum()[df.isna().sum() > 0])

# ==============================================================================
# 2. DATA CLEANING & STRATIFIED TRAIN-TEST SPLIT (80:20)
# ==============================================================================
print("\n" + "=" * 70)
print("2. DATA CLEANING & TRAIN-TEST SPLIT")
print("=" * 70)

# Remove exact duplicates
dup_count = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f"Removed {dup_count} duplicate rows. Remaining rows: {len(df)}")

# Encode target variable: <=50K -> 0, >50K -> 1
target_col = "income"
X = df.drop(columns=[target_col])
y = df[target_col].map({"<=50K": 0, ">50K": 1})

print("\nTarget Class Distribution (%):")
print((y.value_counts(normalize=True) * 100).round(2))

# 80:20 Stratified split with fixed random seed
RANDOM_SEED = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)

print(f"\nTraining set shape: {X_train.shape}")
print(f"Testing set shape : {X_test.shape}")

# Save test_data.csv for GitHub repository requirement
test_export = X_test.copy()
test_export[target_col] = y_test.map({0: "<=50K", 1: ">50K"})
test_export.to_csv("test_data.csv", index=False)
print("Saved 20% test split to 'test_data.csv'")

# ==============================================================================
# 3. PREPROCESSING PIPELINE (FIT ONLY ON TRAINING DATA)
# ==============================================================================
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print(f"\nNumerical features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Fit ONLY on training data, then transform both train and test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Save fitted preprocessor
joblib.dump(preprocessor, "model/preprocessor.joblib")
print(f"Processed feature matrix shape: {X_train_processed.shape}")

# ==============================================================================
# 4. MODEL TRAINING & EVALUATION (5 CLASSIFICATION MODELS)
# ==============================================================================
print("\n" + "=" * 70)
print("3. TRAINING & EVALUATING MODELS")
print("=" * 70)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=RANDOM_SEED),
    "kNN": KNeighborsClassifier(n_neighbors=11, metric="minkowski", p=2),
    "Naive Bayes": GaussianNB(),
    "Random Forest (Ensemble)": RandomForestClassifier(
        n_estimators=150, max_depth=16, random_state=RANDOM_SEED, n_jobs=-1
    )
}

results = []

for name, model in models.items():
    # Train model
    model.fit(X_train_processed, y_train)

    # Predictions & Probabilities on Test Set
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]

    # Calculate all 6 required metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append({
        "ML Model Name": name,
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4)
    })

    # Save trained model artifact inside model/ folder
    safe_filename = name.lower().replace(" ", "_").replace("(", "").replace(")", "") + ".joblib"
    joblib.dump(model, os.path.join("model", safe_filename))

results_df = pd.DataFrame(results)
print("\nEVALUATION METRICS COMPARISON TABLE:")
print(results_df.to_string(index=False))

# Save metrics table to CSV for reference
results_df.to_csv("model/evaluation_metrics.csv", index=False)

# ==============================================================================
# 5. VISUALIZATION OF METRICS (GRAPH REPRESENTATION)
# ==============================================================================
print("\n" + "=" * 70)
print("4. GENERATING METRIC COMPARISON CHARTS")
print("=" * 70)

sns.set_theme(style="whitegrid")
metrics_list = ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]

# Chart 1: Grouped Bar Chart comparing all 6 metrics across all 5 models
df_melted = results_df.melt(
    id_vars="ML Model Name",
    value_vars=metrics_list,
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(13, 6))
ax = sns.barplot(
    data=df_melted,
    x="Metric",
    y="Score",
    hue="ML Model Name",
    palette="viridis"
)
plt.title("Classification Models Performance Comparison Across All 6 Metrics", fontsize=14, fontweight="bold")
plt.ylim(0, 1.05)
plt.ylabel("Score", fontsize=12)
plt.xlabel("Evaluation Metric", fontsize=12)
plt.legend(title="ML Model", bbox_to_anchor=(1.01, 1), loc="upper left")

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=7.5, padding=2, rotation=90)

plt.tight_layout()
plt.savefig("metrics_comparison_bar.png", dpi=300)
plt.close()

# Chart 2: Heatmap of Model vs. Metrics for quick visual ranking
plt.figure(figsize=(10, 5))
heatmap_data = results_df.set_index("ML Model Name")
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".4f",
    cmap="YlGnBu",
    cbar_kws={"label": "Metric Score"},
    linewidths=0.5
)
plt.title("Model Evaluation Metrics Heatmap (Adult Census Dataset)", fontsize=13, fontweight="bold")
plt.ylabel("")
plt.tight_layout()
plt.savefig("metrics_comparison_heatmap.png", dpi=300)
plt.close()

print("Saved 'metrics_comparison_bar.png' and 'metrics_comparison_heatmap.png'")
print("All trained models saved inside 'model/' folder.")
print("Execution completed successfully!")

1. DATASET OVERVIEW
Initial Shape: 32561 rows, 15 columns

Missing values per column (after converting '?' to NaN):
workclass         1836
occupation        1843
native.country     583
dtype: int64

2. DATA CLEANING & TRAIN-TEST SPLIT
Removed 24 duplicate rows. Remaining rows: 32537

Target Class Distribution (%):
income
0    75.91
1    24.09
Name: proportion, dtype: float64

Training set shape: (26029, 14)
Testing set shape : (6508, 14)
Saved 20% test split to 'test_data.csv'

Numerical features (6): ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']
Categorical features (8): ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country']
Processed feature matrix shape: (26029, 104)

3. TRAINING & EVALUATING MODELS

EVALUATION METRICS COMPARISON TABLE:
           ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
     Logistic Regression    0.8502 0.9004     0.7344  0.5925 0.6558 0.5668
        